<a href="https://colab.research.google.com/github/ps-research/The-Language-of-AI-Liability/blob/main/Notebook%200%20%3A%20Data%20Sanity%20Checker%20and%20Fixer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import re
import json
from pathlib import Path
from collections import defaultdict

In [2]:
# ── Configuration ──
BASE_DIR = Path("/content/drive/MyDrive/ACL SRW TEXTS")

FILES = [
    "eu_ai_act.txt",
    "china_genai_measures.txt",
    "south_korea_ai_basic_act.txt",
    "ca_sb1047.txt",
    "ca_sb53.txt",
    "ca_sb942.txt",
    "ca_ab2013.txt",
    "co_sb205.txt",
    "tx_traiga.txt",
    "il_hb3773.txt",
]

In [3]:
# ── Check 1: File Existence & Encoding ──
def check_encoding_and_existence(filepath):
    """Try reading as UTF-8. Report success or failure."""
    issues = []
    if not filepath.exists():
        return None, ["FILE NOT FOUND"]
    try:
        text = filepath.read_text(encoding="utf-8")
    except UnicodeDecodeError as e:
        issues.append(f"UTF-8 DECODE ERROR: {e}")
        # Attempt fallback read to still run other checks
        text = filepath.read_text(encoding="latin-1")
        issues.append("Fell back to latin-1 for remaining checks")
    return text, issues

In [4]:
# ── Check 2: HTML Entities & Artifacts ──
HTML_ENTITIES = re.compile(
    r'&(?:amp|lt|gt|quot|apos|sect|nbsp|mdash|ndash|rsquo|lsquo|rdquo|ldquo'
    r'|hellip|copy|reg|trade|frac\d\d|#\d+|#x[0-9a-fA-F]+);'
)

HTML_TAGS = re.compile(r'</?(?:p|div|span|br|table|tr|td|th|a|b|i|u|em|strong|h[1-6]|li|ul|ol|img|section|article|header|footer|nav|main)\b[^>]*>', re.IGNORECASE)

def check_html_artifacts(text):
    issues = []
    # HTML entities
    entities = HTML_ENTITIES.findall(text)
    if entities:
        counts = defaultdict(int)
        for e in entities:
            counts[e] += 1
        issues.append(f"HTML entities found: {dict(counts)}")

    # Residual HTML tags
    tags = HTML_TAGS.findall(text)
    if tags:
        unique_tags = set(t.lower().split()[0].strip('<>/') for t in tags[:20])
        issues.append(f"HTML tags found ({len(tags)} total): sample tags = {unique_tags}")

    # Common Unicode replacement artifacts
    if '\ufffd' in text:
        count = text.count('\ufffd')
        issues.append(f"Unicode replacement character U+FFFD found {count} times (encoding issue)")

    # Escaped unicode that shouldn't be there
    escaped = re.findall(r'\\u[0-9a-fA-F]{4}', text)
    if escaped:
        issues.append(f"Escaped unicode sequences found: {set(escaped[:10])}")

    return issues

In [5]:
# ── Check 3: Section Markers ──
# Jurisdiction-specific patterns
SECTION_MARKERS = {
    "eu_ai_act.txt": {
        "Article": re.compile(r'\bArticle\s+\d+', re.IGNORECASE),
        "Chapter": re.compile(r'\bChapter\s+[IVXLCDM\d]+', re.IGNORECASE),
        "Annex": re.compile(r'\bAnnex\s+[IVXLCDM]+', re.IGNORECASE),
        "Recital": re.compile(r'^\s*\(\d+\)', re.MULTILINE),
    },
    "china_genai_measures.txt": {
        "Article": re.compile(r'\bArticle\s+\d+', re.IGNORECASE),
        "Chapter": re.compile(r'\bChapter\s+[IVXLCDM\d]+', re.IGNORECASE),
    },
    "south_korea_ai_basic_act.txt": {
        "Article": re.compile(r'\bArticle\s+\d+', re.IGNORECASE),
        "Chapter": re.compile(r'\bChapter\s+[IVXLCDM\d]+', re.IGNORECASE),
        "Section": re.compile(r'\bSection\s+\d+', re.IGNORECASE),
    },
    "ca_sb1047.txt": {
        "SEC.": re.compile(r'\bSEC(?:TION)?\.?\s+\d+', re.IGNORECASE),
        "Section_ref": re.compile(r'\bSection\s+\d+[\.\d]*', re.IGNORECASE),
        "Subdivision": re.compile(r'\b(?:subdivision|paragraph|subparagraph)\s*\([a-zA-Z0-9]+\)', re.IGNORECASE),
    },
    "ca_sb53.txt": {
        "SEC.": re.compile(r'\bSEC(?:TION)?\.?\s+\d+', re.IGNORECASE),
        "Section_ref": re.compile(r'\bSection\s+\d+[\.\d]*', re.IGNORECASE),
        "Subdivision": re.compile(r'\b(?:subdivision|paragraph|subparagraph)\s*\([a-zA-Z0-9]+\)', re.IGNORECASE),
    },
    "ca_sb942.txt": {
        "SEC.": re.compile(r'\bSEC(?:TION)?\.?\s+\d+', re.IGNORECASE),
        "Section_ref": re.compile(r'\bSection\s+\d+[\.\d]*', re.IGNORECASE),
    },
    "ca_ab2013.txt": {
        "SEC.": re.compile(r'\bSEC(?:TION)?\.?\s+\d+', re.IGNORECASE),
        "Section_ref": re.compile(r'\bSection\s+\d+[\.\d]*', re.IGNORECASE),
    },
    "co_sb205.txt": {
        "SECTION": re.compile(r'\bSECTION\s+\d+', re.IGNORECASE),
        "CRS_ref": re.compile(r'\b\d+-\d+-\d+', re.IGNORECASE),  # Colorado Revised Statutes format
    },
    "tx_traiga.txt": {
        "Sec.": re.compile(r'\bSec\.\s*\d+[\.\d]*', re.IGNORECASE),
        "SECTION": re.compile(r'\bSECTION\s+\d+', re.IGNORECASE),
        "Chapter": re.compile(r'\bChapter\s+\d+', re.IGNORECASE),
    },
    "il_hb3773.txt": {
        "Section": re.compile(r'\bSection\s+\d+[-\.\d]*', re.IGNORECASE),
        "Article": re.compile(r'\bArticle\s+\d+', re.IGNORECASE),
    },
}

In [6]:
def check_section_markers(filename, text):
    markers = SECTION_MARKERS.get(filename, {})
    results = {}
    for name, pattern in markers.items():
        matches = pattern.findall(text)
        results[name] = {
            "count": len(matches),
            "samples": matches[:5]  # Show first 5 matches
        }
    return results

In [7]:
# ── Check 4: Basic Statistics ──
def basic_stats(text):
    words = text.split()
    lines = text.split('\n')
    non_empty_lines = [l for l in lines if l.strip()]
    # Rough sentence count (period/question/exclamation followed by space or newline)
    rough_sentences = len(re.findall(r'[.!?](?:\s|$)', text))
    return {
        "word_count": len(words),
        "line_count": len(lines),
        "non_empty_lines": len(non_empty_lines),
        "character_count": len(text),
        "rough_sentence_count": rough_sentences,
    }


# ── Check 5: Quick Deontic Modal Spot Check ──
DEONTIC_QUICK = {
    "shall": re.compile(r'\bshall\b', re.IGNORECASE),
    "shall not": re.compile(r'\bshall\s+not\b', re.IGNORECASE),
    "must": re.compile(r'\bmust\b', re.IGNORECASE),
    "must not": re.compile(r'\bmust\s+not\b', re.IGNORECASE),
    "may": re.compile(r'\bmay\b', re.IGNORECASE),
    "should": re.compile(r'\bshould\b', re.IGNORECASE),
    "is required to": re.compile(r'\bis\s+required\s+to\b', re.IGNORECASE),
    "is prohibited": re.compile(r'\bis\s+prohibited\b', re.IGNORECASE),
}

def quick_deontic_count(text):
    return {modal: len(pattern.findall(text)) for modal, pattern in DEONTIC_QUICK.items()}

In [8]:
# ═══════════════════════════════════════
# ── MAIN DIAGNOSTIC LOOP ──
# ═══════════════════════════════════════

print("=" * 70)
print("CORPUS SANITY CHECK — ACL SRW 2026")
print("=" * 70)

all_results = {}

for filename in FILES:
    filepath = BASE_DIR / filename
    print(f"\n{'─' * 70}")
    print(f"  {filename}")
    print(f"{'─' * 70}")

    # Encoding check
    text, encoding_issues = check_encoding_and_existence(filepath)
    if text is None:
        print("  ❌ FILE NOT FOUND — check path and filename")
        all_results[filename] = {"status": "MISSING"}
        continue

    if encoding_issues:
        for issue in encoding_issues:
            print(f"  ⚠️  ENCODING: {issue}")
    else:
        print("  ✅ UTF-8 encoding OK")

    # HTML artifacts
    html_issues = check_html_artifacts(text)
    if html_issues:
        for issue in html_issues:
            print(f"  ⚠️  HTML: {issue}")
    else:
        print("  ✅ No HTML entities or tags detected")

    # Basic stats
    stats = basic_stats(text)
    print(f"  📊 Words: {stats['word_count']:,} | "
          f"Lines: {stats['non_empty_lines']:,} | "
          f"Chars: {stats['character_count']:,} | "
          f"~Sentences: {stats['rough_sentence_count']:,}")

    # Section markers
    markers = check_section_markers(filename, text)
    print(f"  📌 Section markers:")
    for name, data in markers.items():
        status = "✅" if data["count"] > 0 else "⚠️  NONE FOUND"
        samples = ", ".join(data["samples"][:3]) if data["samples"] else "—"
        print(f"     {status} {name}: {data['count']} found — e.g. [{samples}]")

    # Quick deontic count
    deontics = quick_deontic_count(text)
    total_deontic = sum(deontics.values())
    print(f"  📋 Deontic modals (raw count, pre-pipeline):")
    modal_str = " | ".join(f"{k}: {v}" for k, v in deontics.items() if v > 0)
    print(f"     {modal_str}")
    if total_deontic == 0:
        print(f"  ⚠️  NO DEONTIC MODALS FOUND — check if text was extracted correctly")

    # Store results
    all_results[filename] = {
        "status": "OK" if not encoding_issues and not html_issues else "ISSUES",
        "encoding_issues": encoding_issues,
        "html_issues": html_issues,
        "stats": stats,
        "section_markers": {k: v["count"] for k, v in markers.items()},
        "deontic_counts": deontics,
    }

CORPUS SANITY CHECK — ACL SRW 2026

──────────────────────────────────────────────────────────────────────
  eu_ai_act.txt
──────────────────────────────────────────────────────────────────────
  ✅ UTF-8 encoding OK
  ✅ No HTML entities or tags detected
  📊 Words: 90,044 | Lines: 2,768 | Chars: 598,603 | ~Sentences: 2,545
  📌 Section markers:
     ✅ Article: 576 found — e.g. [Article 114, Article 16, Article 16]
     ✅ Chapter: 69 found — e.g. [Chapter 2, Chapter 4, Chapter 5]
     ✅ Annex: 130 found — e.g. [Annex I, Annex I, Annex II]
     ✅ Recital: 254 found — e.g. [
 

(1), 
 

(2), 
 

(3)]
  📋 Deontic modals (raw count, pre-pipeline):
     shall: 868 | shall not: 51 | must: 3 | may: 268 | should: 519 | is required to: 1

──────────────────────────────────────────────────────────────────────
  china_genai_measures.txt
──────────────────────────────────────────────────────────────────────
  ✅ UTF-8 encoding OK
  ✅ No HTML entities or tags detected
  📊 Words: 1,991 | Lines: 60 | Cha

In [9]:
# ═══════════════════════════════════════
# ── SUMMARY TABLE ──
# ═══════════════════════════════════════

print(f"\n\n{'=' * 70}")
print("SUMMARY")
print(f"{'=' * 70}")
print(f"{'File':<35} {'Words':>8} {'~Sents':>8} {'Deontic':>8} {'Status':>10}")
print(f"{'─' * 35} {'─' * 8} {'─' * 8} {'─' * 8} {'─' * 10}")

for filename in FILES:
    r = all_results.get(filename, {})
    if r.get("status") == "MISSING":
        print(f"{filename:<35} {'—':>8} {'—':>8} {'—':>8} {'MISSING':>10}")
        continue
    s = r.get("stats", {})
    d = r.get("deontic_counts", {})
    total_d = sum(d.values())
    status = r.get("status", "?")
    print(f"{filename:<35} {s.get('word_count', 0):>8,} "
          f"{s.get('rough_sentence_count', 0):>8,} "
          f"{total_d:>8} "
          f"{'✅' if status == 'OK' else '⚠️':>10}")

# ── Save results to JSON for reference ──
output_path = BASE_DIR / "sanity_check_results.json"
with open(output_path, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nDetailed results saved to: {output_path}")



SUMMARY
File                                   Words   ~Sents  Deontic     Status
─────────────────────────────────── ──────── ──────── ──────── ──────────
eu_ai_act.txt                         90,044    2,545     1710          ✅
china_genai_measures.txt               1,991       43       41          ✅
south_korea_ai_basic_act.txt          10,166      504      203          ✅
ca_sb1047.txt                          8,223      259      101          ✅
ca_sb53.txt                            5,881      205       98          ✅
ca_sb942.txt                           1,199       55       18          ✅
ca_ab2013.txt                            829       30        6          ✅
co_sb205.txt                           6,106      110       45          ✅
tx_traiga.txt                          5,706      234       71          ✅
il_hb3773.txt                          3,887       72       18          ✅

Detailed results saved to: /content/drive/MyDrive/ACL SRW TEXTS/sanity_check_results.json


---
---
---
---

In [10]:
"""
Fix Colorado newline issue + Check EU AI Act recital/article boundary
"""

import re
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/ACL SRW TEXTS")

# ═══════════════════════════════════════
# FIX 1: Colorado — collapse stray newlines in SECTION markers
# ═══════════════════════════════════════

co_path = BASE_DIR / "co_sb205.txt"
co_text = co_path.read_text(encoding="utf-8")

# Show the problem first
print("COLORADO — BEFORE FIX:")
print("─" * 50)
problem_matches = re.findall(r'SECTION\s*\n\s*\d+', co_text)
print(f"Found {len(problem_matches)} SECTION markers with stray newlines:")
for m in problem_matches[:5]:
    print(f"  [{repr(m)}]")

# Fix: collapse newline between SECTION and its number
co_fixed = re.sub(r'(SECTION)\s*\n\s*(\d+)', r'\1 \2', co_text)

# Verify fix worked
print(f"\nCOLORADO — AFTER FIX:")
print("─" * 50)
clean_matches = re.findall(r'SECTION\s+\d+', co_fixed)
print(f"Clean SECTION markers now: {len(clean_matches)}")
for m in clean_matches[:8]:
    print(f"  [{m}]")

# Save fixed version
co_path.write_text(co_fixed, encoding="utf-8")
print(f"\n✅ Fixed file saved to {co_path}")


# ═══════════════════════════════════════
# CHECK 2: EU AI Act — where do recitals end and articles begin?
# ═══════════════════════════════════════

print(f"\n\n{'═' * 60}")
print("EU AI ACT — STRUCTURE CHECK")
print(f"{'═' * 60}")

eu_path = BASE_DIR / "eu_ai_act.txt"
eu_text = eu_path.read_text(encoding="utf-8")

# Find all recital markers (numbered paragraphs at the start)
recital_positions = [(m.start(), m.group()) for m in re.finditer(r'^\s*\(\d+\)', eu_text, re.MULTILINE)]

# Find first Article marker
article_positions = [(m.start(), m.group()) for m in re.finditer(r'\bArticle\s+\d+\b', eu_text)]

# Find Annex markers
annex_positions = [(m.start(), m.group()) for m in re.finditer(r'\bAnnex\s+[IVXLCDM]+\b', eu_text)]

print(f"\nRecitals: {len(recital_positions)} found")
if recital_positions:
    print(f"  First recital at char {recital_positions[0][0]}: {recital_positions[0][1]}")
    print(f"  Last recital at char {recital_positions[-1][0]}: {recital_positions[-1][1]}")

print(f"\nArticles: {len(article_positions)} found")
if article_positions:
    print(f"  First article at char {article_positions[0][0]}: {article_positions[0][1]}")
    print(f"  Last article at char {article_positions[-1][0]}: {article_positions[-1][1]}")

print(f"\nAnnexes: {len(annex_positions)} found")
if annex_positions:
    first_annex = annex_positions[0]
    print(f"  First annex at char {first_annex[0]}: {first_annex[1]}")

# Check if there's a clean boundary
if recital_positions and article_positions:
    last_recital_pos = recital_positions[-1][0]
    first_article_pos = article_positions[0][0]

    if first_article_pos > last_recital_pos:
        print(f"\n✅ Clean boundary: recitals end at ~char {last_recital_pos}, "
              f"articles begin at ~char {first_article_pos}")

        # Show what's between the last recital and first article
        boundary_text = eu_text[last_recital_pos:first_article_pos]
        # Trim to something readable
        boundary_clean = boundary_text.strip()[:500]
        print(f"\nText between last recital and first article (first 500 chars):")
        print(f"  ...{boundary_clean}...")
    else:
        print(f"\n⚠️  Recitals and articles are interleaved — boundary unclear")

# Count deontic modals in recitals vs articles
if recital_positions and article_positions:
    recital_zone = eu_text[:first_article_pos]
    article_zone = eu_text[first_article_pos:]

    # If there are annexes, split articles from annexes
    if annex_positions:
        first_annex_pos = annex_positions[0][0]
        if first_annex_pos > first_article_pos:
            article_zone = eu_text[first_article_pos:first_annex_pos]
            annex_zone = eu_text[first_annex_pos:]
        else:
            annex_zone = ""
    else:
        annex_zone = ""

    print(f"\n{'─' * 50}")
    print(f"DEONTIC MODAL DISTRIBUTION BY ZONE:")
    print(f"{'─' * 50}")

    for zone_name, zone_text in [("Recitals", recital_zone),
                                  ("Articles", article_zone),
                                  ("Annexes", annex_zone)]:
        if not zone_text:
            continue
        words = len(zone_text.split())
        shall_n = len(re.findall(r'\bshall\b', zone_text, re.IGNORECASE))
        should_n = len(re.findall(r'\bshould\b', zone_text, re.IGNORECASE))
        may_n = len(re.findall(r'\bmay\b', zone_text, re.IGNORECASE))
        must_n = len(re.findall(r'\bmust\b', zone_text, re.IGNORECASE))
        print(f"\n  {zone_name} (~{words:,} words):")
        print(f"    shall: {shall_n} | should: {should_n} | "
              f"may: {may_n} | must: {must_n}")

COLORADO — BEFORE FIX:
──────────────────────────────────────────────────
Found 7 SECTION markers with stray newlines:
  ['SECTION\n6']
  ['SECTION\n7']
  ['SECTION\n6']
  ['SECTION\n6']
  ['SECTION\n6']

COLORADO — AFTER FIX:
──────────────────────────────────────────────────
Clean SECTION markers now: 29
  [SECTION 1]
  [SECTION 6]
  [SECTION 7]
  [SECTION 6]
  [SECTION 6]
  [SECTION 6]
  [SECTION 6]
  [SECTION 6]

✅ Fixed file saved to /content/drive/MyDrive/ACL SRW TEXTS/co_sb205.txt


════════════════════════════════════════════════════════════
EU AI ACT — STRUCTURE CHECK
════════════════════════════════════════════════════════════

Recitals: 254 found
  First recital at char 1167: 
 

(1)
  Last recital at char 519507: 
 

(6)

Articles: 575 found
  First article at char 3554: Article 114
  Last article at char 597081: Article 51

Annexes: 117 found
  First annex at char 167173: Annex I

⚠️  Recitals and articles are interleaved — boundary unclear

───────────────────────────────

---
---
---

In [11]:
"""
EU AI Act — Corrected structure boundary detection
The real boundary is CHAPTER I / Article 1, not the first 'Article' reference
"""

import re
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/ACL SRW TEXTS")
eu_path = BASE_DIR / "eu_ai_act.txt"
eu_text = eu_path.read_text(encoding="utf-8")

# ── Find the real boundary: "CHAPTER I" ──
chapter1_matches = list(re.finditer(r'CHAPTER\s+I\b', eu_text))
print(f"'CHAPTER I' occurrences: {len(chapter1_matches)}")
for m in chapter1_matches[:5]:
    context = eu_text[m.start():m.start()+200].replace('\n', ' ').strip()
    print(f"  char {m.start()}: ...{context[:150]}...")

# ── Find "Article 1" (the actual first article, not a treaty reference) ──
art1_matches = list(re.finditer(r'\bArticle\s+1\b', eu_text))
print(f"\n'Article 1' occurrences: {len(art1_matches)}")
for m in art1_matches[:5]:
    context = eu_text[m.start():m.start()+200].replace('\n', ' ').strip()
    print(f"  char {m.start()}: ...{context[:120]}...")

# ── Use CHAPTER I as the boundary ──
if chapter1_matches:
    # The first CHAPTER I that's followed by GENERAL PROVISIONS or Article 1
    boundary_pos = chapter1_matches[0].start()
    print(f"\n{'═' * 60}")
    print(f"Using CHAPTER I at char {boundary_pos} as recital/article boundary")
    print(f"{'═' * 60}")

    # Find first Annex
    annex_matches = list(re.finditer(r'\bANNEX\s+[IVXLCDM]+\b', eu_text))
    if not annex_matches:
        # Try mixed case
        annex_matches = list(re.finditer(r'\bAnnex\s+[IVXLCDM]+\b', eu_text))

    # Split into zones
    recital_zone = eu_text[:boundary_pos]

    if annex_matches:
        first_annex_pos = annex_matches[0].start()
        article_zone = eu_text[boundary_pos:first_annex_pos]
        annex_zone = eu_text[first_annex_pos:]
        print(f"First Annex at char {first_annex_pos}")
    else:
        article_zone = eu_text[boundary_pos:]
        annex_zone = ""
        first_annex_pos = None

    # ── Report zone sizes ──
    print(f"\nZone sizes:")
    print(f"  Recitals:  {len(recital_zone.split()):>8,} words  "
          f"(chars 0 – {boundary_pos:,})")
    print(f"  Articles:  {len(article_zone.split()):>8,} words  "
          f"(chars {boundary_pos:,} – {first_annex_pos:,})" if first_annex_pos
          else f"  Articles:  {len(article_zone.split()):>8,} words")
    if annex_zone:
        print(f"  Annexes:   {len(annex_zone.split()):>8,} words  "
              f"(chars {first_annex_pos:,} – end)")

    # ── Deontic modal distribution by zone ──
    print(f"\n{'─' * 60}")
    print(f"DEONTIC MODAL DISTRIBUTION BY ZONE:")
    print(f"{'─' * 60}")

    for zone_name, zone_text in [("Recitals", recital_zone),
                                  ("Articles (Ch I–XIII)", article_zone),
                                  ("Annexes (I–XIII)", annex_zone)]:
        if not zone_text:
            continue
        words = len(zone_text.split())
        shall_c = len(re.findall(r'\bshall\b', zone_text, re.IGNORECASE))
        shall_not_c = len(re.findall(r'\bshall\s+not\b', zone_text, re.IGNORECASE))
        should_c = len(re.findall(r'\bshould\b', zone_text, re.IGNORECASE))
        may_c = len(re.findall(r'\bmay\b', zone_text, re.IGNORECASE))
        may_not_c = len(re.findall(r'\bmay\s+not\b', zone_text, re.IGNORECASE))
        must_c = len(re.findall(r'\bmust\b', zone_text, re.IGNORECASE))

        print(f"\n  {zone_name} (~{words:,} words):")
        print(f"    shall: {shall_c:>4}  |  shall not: {shall_not_c:>3}  |  "
              f"should: {should_c:>4}  |  may: {may_c:>4}  |  "
              f"may not: {may_not_c:>3}  |  must: {must_c:>3}")

        # Ratios
        total_deontic = shall_c + should_c + may_c + must_c
        if total_deontic > 0:
            print(f"    → shall: {shall_c/total_deontic:.1%}  "
                  f"should: {should_c/total_deontic:.1%}  "
                  f"may: {may_c/total_deontic:.1%}  "
                  f"must: {must_c/total_deontic:.1%}")

    # ── Verify: count recital markers (1)–(180) in the recital zone only ──
    recital_markers = re.findall(r'^\s*\(\d+\)', recital_zone, re.MULTILINE)
    print(f"\n{'─' * 60}")
    print(f"Recital markers found in recital zone: {len(recital_markers)}")
    if recital_markers:
        print(f"  First: {recital_markers[0].strip()}  Last: {recital_markers[-1].strip()}")

    # ── Count articles in article zone ──
    articles_in_zone = re.findall(r'\bArticle\s+(\d+)\b', article_zone)
    if articles_in_zone:
        nums = sorted(set(int(a) for a in articles_in_zone))
        print(f"Articles found in article zone: {len(set(articles_in_zone))} unique")
        print(f"  Range: Article {nums[0]} – Article {nums[-1]}")

'CHAPTER I' occurrences: 1
  char 230927: ...CHAPTER I  GENERAL PROVISIONS  Article 1  Subject matter\`  1.   The purpose of this Regulation is to improve the functioning of the internal market a...

'Article 1' occurrences: 1
  char 230958: ...Article 1  Subject matter\`  1.   The purpose of this Regulation is to improve the functioning of the internal market an...

════════════════════════════════════════════════════════════
Using CHAPTER I at char 230927 as recital/article boundary
════════════════════════════════════════════════════════════
First Annex at char 548881

Zone sizes:
  Recitals:    34,717 words  (chars 0 – 230,927)
  Articles:    47,935 words  (chars 230,927 – 548,881)
  Annexes:      7,392 words  (chars 548,881 – end)

────────────────────────────────────────────────────────────
DEONTIC MODAL DISTRIBUTION BY ZONE:
────────────────────────────────────────────────────────────

  Recitals (~34,717 words):
    shall:    0  |  shall not:   0  |  should:  515  |  may:  101 

---
---

In [12]:
"""
EU AI Act — Save the binding provisions (Articles + Annexes),
dropping the non-binding recitals.
Keep the original file intact for reference.
"""

import re
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/ACL SRW TEXTS")
eu_path = BASE_DIR / "eu_ai_act.txt"
eu_text = eu_path.read_text(encoding="utf-8")

# Find CHAPTER I boundary
chapter1 = re.search(r'CHAPTER\s+I\b', eu_text)
boundary = chapter1.start()

# Split
recitals = eu_text[:boundary]
binding = eu_text[boundary:]  # Articles + Annexes

# Save binding provisions as the working version
binding_path = BASE_DIR / "eu_ai_act_binding.txt"
binding_path.write_text(binding, encoding="utf-8")

# Keep original intact but also save recitals separately for reference
recitals_path = BASE_DIR / "eu_ai_act_recitals.txt"
recitals_path.write_text(recitals, encoding="utf-8")

# Verify
print(f"Original full text:     {len(eu_text.split()):>8,} words")
print(f"Recitals (saved sep.):  {len(recitals.split()):>8,} words → {recitals_path.name}")
print(f"Binding provisions:     {len(binding.split()):>8,} words → {binding_path.name}")
print(f"\nBinding text starts with:")
print(f"  {binding[:200].strip()}")
print(f"\n✅ Use 'eu_ai_act_binding.txt' in the pipeline")
print(f"   Keep 'eu_ai_act.txt' as the original for reference")
print(f"   'eu_ai_act_recitals.txt' saved separately for optional analysis")

Original full text:       90,044 words
Recitals (saved sep.):    34,717 words → eu_ai_act_recitals.txt
Binding provisions:       55,327 words → eu_ai_act_binding.txt

Binding text starts with:
  CHAPTER I

GENERAL PROVISIONS

Article 1

Subject matter\`

1.   The purpose of this Regulation is to improve the functioning of the internal market and promote the uptake of human-centric and trustwo

✅ Use 'eu_ai_act_binding.txt' in the pipeline
   Keep 'eu_ai_act.txt' as the original for reference
   'eu_ai_act_recitals.txt' saved separately for optional analysis


---
---